# Modelagem NPS — definição do target

Este notebook implementa as verificações da Seção **4.3.2.2** de `documents/documentacao.md`. Ele consome a base analítica já processada e não integra, modifica ou sobrescreve arquivos em `data/raw/`.

> **Confidencialidade:** os dados e quaisquer saídas produzidas pela execução permanecem locais. O notebook é versionado sem outputs de células.

O escopo desta etapa é exclusivamente definir e validar o alvo binário `DETRATOR`, registrar o desbalanceamento e conferir as colunas candidatas a *leakage*. Não há treinamento, engenharia de atributos, reamostragem ou alteração da unidade de análise.

## 1. Contrato de modelagem

A pontuação ocorre após o encerramento operacional da jornada e antes da resposta à pesquisa NPS. Cada linha representa uma resposta identificada por `RESPONDENT_ID`; jornadas com conexão permanecem uma única observação. O modelo deve estimar `P(DETRATOR = 1 | X)` somente a partir de atributos disponíveis nesse instante.

## Conexão com a base de dados

As bases da Azul **não são versionadas** neste repositório, por compromisso de proteção de dados firmado entre o Inteli e o parceiro: a pasta `data/` e as extensões `.csv`, `.xlsx` e `.parquet` estão no `.gitignore`. Para executar este notebook:

1. **Arquivos brutos.** Copie os arquivos `.csv` recebidos da Azul para `data/raw/`, na raiz do repositório:
   - `PROJETO_INTELI.NPS_01.csv` a `PROJETO_INTELI.NPS_04.csv`;
   - `PROJETO_INTELI.PERFIL_CLIENTE_01.csv` e `PROJETO_INTELI.PERFIL_CLIENTE_02.csv`;
   - `PROJETO_INTELI.INFORMACAO_VIAGEM.csv`;
   - `PROJETO_INTELI.DISTRIBUICAO_PAX_NORMALIZADO.csv`.

   Deixe fora de `data/raw/` as cópias `.xlsx` das mesmas bases: o pré-processamento compara os dois formatos de uma mesma parte e interrompe a execução se eles divergirem.
2. **Base analítica.** Execute `notebooks/pre-processamento.ipynb`. Ele integra os arquivos de `data/raw/` e grava `data/processed/base_analitica.parquet`, a base que os notebooks de análise e de modelagem leem.
3. **Google Colab.** Siga a seção "No Google Colab" do `README.md`: envie a pasta do projeto ao Google Drive, com as bases em `data/raw/`, e execute antes de tudo uma célula que monta o Drive e entra na pasta `notebooks/` do projeto (`%cd`). Com isso, os caminhos descritos abaixo funcionam sem ajuste, como na execução local.

**Neste notebook:** lê `data/processed/base_analitica.parquet`, gerada pelo passo 2. A primeira célula de código encontra a raiz do projeto procurando as pastas `data/` e `scripts/`, e o caminho fica em `CAMINHO_BASE`.

In [ ]:
from pathlib import Path
import sys

import pandas as pd

RAIZ = next(
    candidato
    for candidato in (Path.cwd(), Path.cwd().parent, Path("/content"))
    if (candidato / "data").exists() and (candidato / "scripts").exists()
)
sys.path.insert(0, str(RAIZ / "scripts"))

from preprocessamento_nps import (
    CODIGOS_NPS_PRINCIPAL_VALIDOS,
    criar_target_detrator,
    selecionar_features_score_pos_viagem,
)

CAMINHO_BASE = RAIZ / "data" / "processed" / "base_analitica.parquet"
df = pd.read_parquet(CAMINHO_BASE)

assert df["RESPONDENT_ID"].is_unique, "A unidade de análise não pode ser alterada."
assert len(df) == df["RESPONDENT_ID"].nunique()


## 2. Variável NPS original

A fonte não disponibiliza a nota NPS bruta de 0 a 10. `NPS_PRINCIPAL` já é a classificação codificada: `-100` para Detrator, `0` para Neutro e `100` para Promotor. A regra tradicional de NPS é, portanto, uma hipótese de interpretação compatível com a codificação recebida, mas não uma transformação aplicada sobre notas brutas.

In [ ]:
nps_principal = df["NPS_PRINCIPAL"]
diagnostico_nps = pd.DataFrame({
    "dtype": [str(nps_principal.dtype)],
    "nulos": [int(nps_principal.isna().sum())],
    "valores_unicos": [int(nps_principal.nunique(dropna=True))],
}, index=["NPS_PRINCIPAL"])
distribuicao_nps = nps_principal.value_counts(dropna=False).sort_index()
valores_inesperados = sorted(
    set(nps_principal.loc[~nps_principal.isin(CODIGOS_NPS_PRINCIPAL_VALIDOS)].dropna())
)

assert not valores_inesperados, f"Códigos NPS inesperados: {valores_inesperados}"
diagnostico_nps, distribuicao_nps


## 3. Target binário

A classe positiva é Detrator. Neutros e Promotores são agrupados como não detratores:

```python
df["DETRATOR"] = (df["NPS_PRINCIPAL"] == -100).astype("int8")
```

A função compartilhada abaixo faz a mesma transformação, mas falha para nulos e códigos fora da escala contratada. Isso evita transformar silenciosamente um código desconhecido em classe 0.

In [ ]:
target = criar_target_detrator(df)

if "DETRATOR" in df.columns:
    assert target.equals(df["DETRATOR"]), (
        "A coluna DETRATOR existente diverge da regra definida para NPS_PRINCIPAL."
    )

df_modelagem = df.assign(DETRATOR=target)
distribuicao_target = target.value_counts().sort_index()
proporcao_target = target.value_counts(normalize=True).sort_index()

assert target.isna().sum() == 0
distribuicao_target, proporcao_target


A distribuição deve ser registrada antes da escolha de métricas e estratégias de modelagem. Nenhuma técnica de SMOTE, sobre-amostragem ou subamostragem é aplicada nesta etapa.

## 4. Auditoria de leakage

As respostas coletadas no questionário existem no mesmo momento do desfecho e não podem compor `X` para o score pós-viagem. A lista é uma regra de auditoria: ela não remove colunas da base analítica. A seleção efetiva de atributos ocorre pela allowlist temporal aprovada no pipeline.

In [ ]:
COLUNAS_LEAKAGE = {
    "NPS_PRINCIPAL", "DETRATOR", "CATEGORIA_NPS",
    "NPS_ATRASO", "NPS_BAGAGEM", "NPS_BAGMAO", "NPS_CANCELAMENTO24H",
    "NPS_CKBALCAO", "NPS_CKMOBILE", "NPS_CKTOTEM", "NPS_CKWEB",
    "NPS_COMISSARIOS", "NPS_CONFORTO", "NPS_EMBARQUE", "NPS_ENTRETENIMENTO",
    "NPS_LIMPEZA", "NPS_PILOTOS", "NPS_RESAGENCIA", "NPS_RESWEB",
    "NPS_SNACKS", "NPS_AZULFID", "NPS_WIFI",
    "SUB_ENTRETENIMENTO1", "SUB_ENTRETENIMENTO2",
    "SUB_FIL_MOTIVOVIAGEM", "SUB_FIL_FREQUENCIAAZUL",
}

candidatas_leakage_presentes = sorted(COLUNAS_LEAKAGE.intersection(df_modelagem.columns))
X, features_aprovadas, features_ausentes = selecionar_features_score_pos_viagem(df_modelagem)

assert not COLUNAS_LEAKAGE.intersection(features_aprovadas)
candidatas_leakage_presentes, features_aprovadas, features_ausentes


## 5. Próxima etapa

Com o target validado, a próxima etapa poderá definir a divisão temporal agrupada por Cliente, as métricas e os modelos candidatos. Essas decisões devem continuar respeitando o contrato de `t_score`; este notebook não as executa.